In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

In [ ]:
from pathlib import Path
from sklearn.preprocessing import normalize
import torch
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from tqdm import tqdm
from collections import defaultdict
from PIL import Image
from lucent.optvis import render, param, transform, objectives
import shelve
import ipywidgets as widgets
from IPython.display import display, clear_output
from PIL import Image
import random

import matplotlib.pyplot as plt
import numpy as np
from lucent.modelzoo import inceptionv1
from PIL import Image
from torch.nn import functional as F
from lucent.optvis import param

from olt.act import InputOutputModelSnapshot
import json

from lucent.modelzoo import inceptionv1
from olt.tfms import transform
from olt.act import InputOutputModelSnapshot
from olt.show import show_single_channel_red_green_black as S
from olt.shards import raw_iter_shards, read_image_shard


base_report_dir = Path("mass-train-reports")
plt.style.use("dark_background")


device = "cpu"
model = inceptionv1(pretrained=True)
model = model.to(device)
model = model.eval()


FLAT_IMAGE_DIR = Path(
    "/Users/hariomnarang/Desktop/personal/hiccup-ide/olt/notebooks/this-and-prev/flat-images"
)

In [ ]:
# df = pd.read_csv("main_df.csv")

In [ ]:
from itertools import batched
kelly_colors = [
    "#F2F3F4",  # white (skip as bg reference)
    "#F3C300", "#875692", "#F38400", "#A1CAF1", "#BE0032",
    "#C2B280", "#848482", "#008856", "#E68FAC", "#0067A5", "#F99379",
    "#604E97", "#F6A600", "#B3446C", "#DCD300", "#882D17", "#8DB600",
    "#654522", "#E25822", "#2B3D26",
]


def _get_ip_acts_for_neuron(ikey, layer_name, y, x):
    timg = transform(Image.open(FLAT_IMAGE_DIR / f"{ikey}.jpeg"))[None]
    return InputOutputModelSnapshot.get_activations(timg, model, [layer_name])[layer_name]["input"][0, :, y, x]

def _get_ip_acts_of_row(row):
    return _get_ip_acts_for_neuron(row.input_image_key, row.layer_name, row.y_position, row.x_position)



def _norm(val):
    orig_shape_single_dim = False
    if len(val.shape) == 1:
        orig_shape_single_dim = True
        val = val.reshape(1, -1)
    res = normalize(val, "l2")
    if orig_shape_single_dim:
        res = res.reshape(-1)
    return res


def get_all_tfmed_images_as_batch(shards_base_dir=Path("image-shards")):
    inet_label_dirs = [p for p in shards_base_dir.glob("*") if p.is_dir()]
    
    test_images = []
    
    for d in tqdm(inet_label_dirs):
        shards = raw_iter_shards(d)
        keys, images = next(read_image_shard(shards[0], 1, {}))
        key, image = keys[0], images[0]
        test_images.append(transform(image))
    return test_images


def get_random_sampled_activations(model, test_images, layer_name, channel, disable_tqdm=False):
    model = model.to("mps")
    noise_acts = []

    for batch in tqdm(list(batched(test_images, 8)), disable=disable_tqdm):
        batch = torch.stack(batch).to("mps")
        act = InputOutputModelSnapshot.get_activations(batch, model, [layer_name])[
            layer_name
        ]["output"]
        act = act[:, channel, :, :].reshape(act.shape[0], -1)
        idx = torch.randint(0, act.shape[1], (act.shape[0],))
        samples = act[torch.arange(8), idx]  # shape [8]
        noise_acts.append(samples)
    noise_acts = torch.cat(noise_acts)
    return noise_acts


def plot_grid(label_by_points, n_rows, n_cols, colors=None, figsize=None, sharey=True, max_points_per_label=200):
    label_by_points = {label: np.random.choice(points, min(len(points), max_points_per_label)) for label, points in label_by_points.items()}
    labels = list(label_by_points.keys())
    if colors is None:
        colors = kelly_colors[1:]
    color_map = {l: colors[i % len(colors)] for i, l in enumerate(labels)}

    if figsize is None:
        figsize = (7 * n_cols, 5 * n_rows)

    fig, axes = plt.subplots(n_rows, n_cols, figsize=figsize, sharey=sharey)
    axes = np.array(axes).reshape(-1)  # flatten regardless of shape

    n_labels = len(labels)
    n_panels = n_rows * n_cols
    labels_per_panel = -(-n_labels // n_panels)  # ceil division

    for i, ax in enumerate(axes):
        panel_labels = labels[i * labels_per_panel : (i + 1) * labels_per_panel]
        for l in panel_labels:
            ys = label_by_points[l]
            ax.scatter(range(len(ys)), ys, color=color_map[l], label=l)
        if panel_labels:
            ax.legend()
    for ax in axes[n_labels:]:
        ax.axis("off")
    plt.tight_layout()
    plt.show()
    return fig, axes



def collect_all_noise_activations(model, all_layers, test_images):
    layer_by_channel_by_noise_acts = defaultdict(dict)
    for layer_name in all_layers:
        n_chans = model.get_submodule(layer_name).weight.shape[0]
        for chan in tqdm(range(n_chans), desc=layer_name):
            layer_by_channel_by_noise_acts[layer_name][str(chan)] = get_random_sampled_activations(
                model, test_images, layer_name, chan, disable_tqdm=True
            )
    return layer_by_channel_by_noise_acts

def get_layer_by_channel_by_label_by_act():
    with open("main_act_dict.pkl", "rb") as f:
        main_act_dict = pickle.load(f)
    
    with open("main_act_ckpt.pkl", "rb") as f:
        incoming_act = pickle.load(f)
    
    
    main_act_defdict = defaultdict(lambda: defaultdict(lambda: defaultdict(list)))
    
    for k1, v1 in main_act_dict.items():
        for k2, v2 in v1.items():
            for k3, v3 in v2.items():
                main_act_defdict[k1][k2][k3] = v3
    
    return main_act_defdict

In [ ]:
# layer_names_4d = [
#     "mixed4d_1x1_pre_relu_conv",
#     "mixed4d_3x3_pre_relu_conv",
#     "mixed4d_pool_reduce_pre_relu_conv",
#     "mixed4d_5x5_pre_relu_conv",
# ]
# our_neuron_layer_name = "mixed4e_1x1_pre_relu_conv"


# all_layers = layer_names_4d + [our_neuron_layer_name]

In [ ]:
TEST_IMAGES = get_all_tfmed_images_as_batch()

# Download base reports and CSV collection

In [ ]:
! mkdir mixed5b-mass-trian-reports

In [ ]:
! cd mixed5b-mass-trian-reports && aws s3 sync s3://narang99-private/lucent-workdir/mass-train-reports/mixed5b_5x5_pre_relu_conv/ ./mixed5b_5x5_pre_relu_conv/

In [ ]:
! cd mixed5b-mass-trian-reports && aws s3 sync s3://narang99-private/lucent-workdir/mass-train-reports/mixed5b_5x5_bottleneck_pre_relu_conv/ ./mixed5b_5x5_bottleneck_pre_relu_conv/

In [ ]:
from pathlib import Path
asset_dump_dir = Path("act_range_analysis_mixed5b")
asset_dump_dir.mkdir(parents=True,exist_ok=True)
base_report_dir = Path("mixed5b-mass-trian-reports")

In [ ]:
# extract reports now
for layer_dir in list(base_report_dir.glob("*")):
    if not layer_dir.is_dir():
        continue
    for report_tgz in tqdm(list(layer_dir.glob("*.tgz")), desc=layer_dir.name):
        ! cd {layer_dir} && tar -xf {report_tgz.name}

In [ ]:
csv_files = list(base_report_dir.rglob("report.csv"))

dfs = []
for f in tqdm(csv_files):
    df = pd.read_csv(f)
    df = df[df.cluster_label != -1].reset_index()
    dfs.append(df)

main_df = pd.concat(dfs)

main_df.head()

In [ ]:
main_df.to_csv(asset_dump_dir / "main_df.csv")

# clustered activations collection

In [ ]:
from collections import defaultdict
import pickle
from tqdm import tqdm
import pandas as pd

from PIL import Image

from olt.act import InputOutputModelSnapshot
from olt.tfms import transform


def collect_acts_for_all_image(
    df, model, all_layers, device, flat_image_dir
):
    all_image_keys = df.input_image_key.unique()
    layer_by_chan_by_cluster_id_by_act = defaultdict(
        lambda: defaultdict(lambda: defaultdict(list))
    )
    for image_key in tqdm(all_image_keys):
        batch = transform(Image.open(flat_image_dir / f"{image_key}.jpeg"))[None].to(device)
        idf = df.loc[[image_key]]

        with torch.no_grad():
            acts = InputOutputModelSnapshot.get_activations(batch, model, all_layers)
    
        for tup in idf.itertuples():
            layer_by_chan_by_cluster_id_by_act[tup.layer_name][str(tup.channel)][
                str(tup.cluster_label)
            ].append(
                acts[tup.layer_name]["output"][
                    0, tup.channel, tup.y_position, tup.x_position
                ].item()
            )

    return layer_by_chan_by_cluster_id_by_act




def dump_layer_by_chan_by_cid_by_act(result, checkpoint_path):
    """Convert nested defaultdict of patch-lists to plain dict of stacked tensors, then save."""
    plain = {}
    for layer_name, by_channel in result.items():
        plain[layer_name] = {}
        for channel, by_cid in by_channel.items():
            plain[layer_name][channel] = {}
            for cid, acts in by_cid.items():
                plain[layer_name][channel][cid] = acts
    with open(checkpoint_path, "wb") as f:
        pickle.dump(plain, f)


In [ ]:
df = pd.read_csv(asset_dump_dir / "main_df.csv")
df = df.set_index("input_image_key", drop=False)

In [ ]:
df.head()

In [ ]:
all_layers = df.layer_name.unique()

In [ ]:
all_layers

In [ ]:
model = model.to("mps")
layer_by_chan_by_cluster_id_by_act = collect_acts_for_all_image(
    df, model, all_layers, "mps", FLAT_IMAGE_DIR
)

In [ ]:
dump_layer_by_chan_by_cid_by_act(layer_by_chan_by_cluster_id_by_act, asset_dump_dir / "layer_by_channel_by_cid_by_acts.pkl")

In [ ]:
import gc
del layer_by_chan_by_cluster_id_by_act
gc.collect()

# Noise collection

In [ ]:
TEST_IMAGES = get_all_tfmed_images_as_batch()

In [ ]:
layer_by_channel_by_noise_acts = collect_all_noise_activations(model, all_layers, TEST_IMAGES)

In [ ]:
import pickle
with open(asset_dump_dir / "layer_by_channel_by_noise_acts.pkl", "wb") as f:
    pickle.dump(layer_by_channel_by_noise_acts, f)

In [ ]:
data = layer_by_channel_by_noise_acts["mixed5b_5x5_pre_relu_conv"]['0']
plt.scatter(range(len(data)), data)
plt.show()

# Collect all cluster points

We would now also like one more visualisation. Given a random pointwise mult for a given neuron, i want the closest cluster to it (we do basic single linkage, find the one with max similarity for us, and return that one).  

For that, we need to store each neuron's cluster by point (we would store the cluster label as a tensor, and patches as a tensor).   

In [ ]:

class ReceptiveFieldOutOfBounds(ValueError):
    """Raised by receptive_block when a computed receptive field falls outside
    its tensor's valid range. A dedicated type (rather than a bare ValueError)
    so callers that recover from this (skip the affected probe/firing/sample —
    see analyser.top_contributing_indices, similarity.get_neuron_closest_cluster,
    feature_viz.dump_feature_viz_asset) catch exactly this condition, not some
    unrelated ValueError raised nearby."""


def receptive_block(i, ksize, stride, padding, input_size=None):
    """
    Returns [start, end) input indices (end=exclusive) that influence
    output position i of a conv layer.

    If input_size is given, raises ReceptiveFieldOutOfBounds when the computed
    range falls outside [0, input_size) instead of clamping it. This module
    always multiplies the sliced patch elementwise against a fixed-shape conv
    weight (see analyser.top_contributing_indices, similarity.get_neuron_closest_cluster),
    so a clamped/narrower-than-ksize patch would just fail later with a shape
    mismatch anyway — and a negative start left unclamped would silently wrap
    around via Python/numpy/torch's negative-index slicing, pulling data from
    the wrong end of the tensor instead of erroring. Every caller that's
    slicing a real tensor should pass input_size (from that tensor's own
    shape) so an out-of-range case fails immediately at the coordinate-math
    step, not later as a confusing shape error or, worse, not at all. This
    currently never fires for the one supported current_layer (a
    1x1/stride-1/pad-0 conv, so y0/x0 can never go negative — see
    olt/CLAUDE.md) — it's here so extending to a new layer with a real kernel
    fails fast instead of silently, and so callers can recover per-atom (see
    ReceptiveFieldOutOfBounds) instead of the whole run dying on one
    boundary position.
    """
    start = i * stride - padding
    end = start + ksize

    if input_size is not None and (start < 0 or end > input_size):
        raise ReceptiveFieldOutOfBounds(
            f"receptive_block: computed range [{start}, {end}) for output index "
            f"{i} (ksize={ksize}, stride={stride}, padding={padding}) falls "
            f"outside the input's valid range [0, {input_size}) — this usually "
            "means a current_layer/dep_layer this module wasn't designed for "
            "(see olt/CLAUDE.md)."
        )

    return start, end


def _receptive_block(i, ksize, stride, padding, input_size=None):
    return receptive_block(i, ksize, stride, padding, input_size)    


In [ ]:
# go through the df, for each input image, capture the input acgtivations
# for each point in the df for this image, get its patch (using receptive field calc)
# then save it with the cluster label lol.  


In [ ]:
df = pd.read_csv(asset_dump_dir / "main_df.csv")
df.layer_name.unique()

In [ ]:
layer_names = df.layer_name.unique()
ikeys = df.input_image_key.unique()

In [ ]:
import gc
from itertools import batched

def _build_lookup_structures(df, model):
    # df = df.set_index("input_image_key", drop=False)
    layer_names = df.layer_name.unique()
    ikeys = df.input_image_key.unique()
    groups = dict(tuple(df.groupby("input_image_key")))
    layer_cache = {ln: model.get_submodule(ln) for ln in layer_names}
    return layer_names, ikeys, groups, layer_cache


def _load_batch(batched_ikeys, flat_image_dir, device="mps"):
    return torch.stack(
        [transform(Image.open(flat_image_dir / f"{ikey}.jpeg")) for ikey in batched_ikeys]
    ).to(device)


def _extract_patches_for_image(rdf, acts, layer_cache, j, result, max_per_cid):
    for tup in rdf.itertuples():
        layer = layer_cache[tup.layer_name]
        y0, y1 = _receptive_block(
            tup.y_position, layer.kernel_size[0], layer.stride[0], layer.padding[0]
        )
        x0, x1 = _receptive_block(
            tup.x_position, layer.kernel_size[1], layer.stride[1], layer.padding[1]
        )
        patch = acts[tup.layer_name]["input"][j, :, y0:y1, x0:x1].clone()
        if len(result[tup.layer_name][tup.channel][tup.cluster_label]) < max_per_cid:
            result[tup.layer_name][tup.channel][tup.cluster_label].append(patch)


def _process_batch(batched_ikeys, model, layer_names, groups, layer_cache, flat_image_dir, result, max_per_cid):
    batch = _load_batch(batched_ikeys, flat_image_dir)
    acts = InputOutputModelSnapshot.get_activations(batch, model, layer_names)
    for j, ikey in enumerate(batched_ikeys):
        rdf = groups[ikey]
        _extract_patches_for_image(rdf, acts, layer_cache, j, result, max_per_cid)

def dump_checkpoint(result, checkpoint_path):
    """Convert nested defaultdict of patch-lists to plain dict of stacked tensors, then save."""
    plain = {}
    for layer_name, by_channel in result.items():
        plain[layer_name] = {}
        for channel, by_cid in by_channel.items():
            plain[layer_name][channel] = {}
            for cid, patches in by_cid.items():
                plain[layer_name][channel][cid] = torch.stack(patches)
    torch.save(plain, checkpoint_path)


def load_checkpoint(checkpoint_path):
    """Load a checkpoint saved by dump_checkpoint, converting back into nested defaultdict
    of patch-lists (unstacking tensors back into lists so accumulation can continue)."""
    plain = torch.load(checkpoint_path)
    result = defaultdict(lambda: defaultdict(lambda: defaultdict(list)))
    for layer_name, by_channel in plain.items():
        for channel, by_cid in by_channel.items():
            for cid, stacked in by_cid.items():
                result[layer_name][channel][cid] = list(stacked.unbind(0))
    return result

def _maybe_checkpoint(result, checkpoint_path, batch_idx, checkpoint_every):
    if checkpoint_path is not None and (batch_idx + 1) % checkpoint_every == 0:
        dump_checkpoint(result, checkpoint_path)


def get_neuron_cluster_patches(model, df, flat_image_dir, checkpoint_path=None, checkpoint_every=100, max_per_cid=100):
    layer_names, ikeys, groups, layer_cache = _build_lookup_structures(df, model)
    result = defaultdict(lambda: defaultdict(lambda: defaultdict(list)))
    batches = list(batched(ikeys, 32))

    with torch.no_grad():
        for i, batched_ikeys in enumerate(tqdm(batches)):
            _process_batch(batched_ikeys, model, layer_names, groups, layer_cache, flat_image_dir, result, max_per_cid)
            _maybe_checkpoint(result, checkpoint_path, i, checkpoint_every)

    if checkpoint_path is not None:
        dump_checkpoint(result, checkpoint_path)
        gc.collect()

    return result


In [ ]:
layer_by_channel_by_cid_by_patches = get_neuron_cluster_patches(model, df, FLAT_IMAGE_DIR, "ckpt_layer_by_channel_by_cid_by_patch.pt", 400, 100)

In [ ]:
%reset out

In [ ]:
# now save it
# layername/channel/cid-by-act easy for now

In [ ]:
dest_dir = asset_dump_dir / "cluster-patches-set"

for layer_name in layer_by_channel_by_cid_by_patches:
    for channel in tqdm(layer_by_channel_by_cid_by_patches[layer_name], desc=layer_name):
        d = dest_dir / layer_name / str(channel) / "cid_by_patches.pt"
        d.parent.mkdir(parents=True, exist_ok=True)
        cid_by_patches = layer_by_channel_by_cid_by_patches[layer_name][channel]
        to_save = {cid: torch.stack(patches) for cid, patches in cid_by_patches.items()}
        torch.save(to_save, d)
        

        

In [ ]:
cid_by_patches = torch.load("cluster-patches-set/mixed4e_1x1_pre_relu_conv/55/cid_by_patches.pt", weights_only=False)

In [ ]:
w = model.get_submodule("mixed4e_1x1_pre_relu_conv").weight[55].reshape(-1).detach().cpu()

In [ ]:
for p in cid_by_patches[61][:4]:
    print(p.shape)

In [ ]:
from olt.show import show_single_channel_red_green_black as S

S([(p.reshape(-1) * w.reshape(-1)).reshape(22,24) for p in cid_by_patches[60][:4]], 20, ncols=4)
plt.show()

In [ ]:
torch.stack(layer_by_channel_by_cid_by_patches["mixed4d_3x3_pre_relu_conv"][10][7]).shape